In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

In [5]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/input/datasets/salader/dogsvscats/train',
    image_size=(96,96),
    batch_size=32
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/input/datasets/salader/dogsvscats/test',
    image_size=(96,96),
    batch_size=32
)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.


In [6]:
base_model = MobileNetV2(
    input_shape=(96,96,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [9]:
mobilenet_model = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        128,
        activation='relu'
    ),

    layers.Dense(
        1,
        activation='sigmoid'
    )
])

In [10]:
mobilenet_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [12]:
history = mobilenet_model.fit(
    train_ds,
    epochs=3,
    validation_data=test_ds
)

Epoch 1/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 98s 156ms/step - accuracy: 0.6948 - loss: 0.5747 - val_accuracy: 0.6546 - val_loss: 0.6110
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 98s 157ms/step - accuracy: 0.7020 - loss: 0.5691 - val_accuracy: 0.6710 - val_loss: 0.6010
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 99s 158ms/step - accuracy: 0.7043 - loss: 0.5608 - val_accuracy: 0.6886 - val_loss: 0.5907


In [13]:
loss, mobilenet_acc = mobilenet_model.evaluate(test_ds)

print("MobileNetV2 Accuracy:", mobilenet_acc)

157/157 ━━━━━━━━━━━━━━━━━━━━ 20s 126ms/step - accuracy: 0.6886 - loss: 0.5907
MobileNetV2 Accuracy: 0.6886000037193298
